In [0]:
%pip install hotel_reservation-0.0.1-py3-none-any.whl

Processing ./hotel_reservation-0.0.1-py3-none-any.whl
INFO: pip is looking at multiple versions of mlflow-skinny[databricks] to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of mlflow-skinny[databricks] to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of databricks-agents to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of google-api-core to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of grpcio-status to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 681.8/681.8 kB 26.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 92.6 MB/s eta 0:00:00
  

In [0]:
%restart_python

In [0]:
# Configure tracking uri
from loguru import logger
from pyspark.sql import SparkSession

from hotel_reservation.config import ProjectConfig, Tags
from hotel_reservation.models.feature_lookup_model import FeatureLookUpModel

# Configure tracking uri
# mlflow.set_tracking_uri("databricks")
# mlflow.set_registry_uri("databricks-uc")

spark = SparkSession.builder.getOrCreate()
tags_dict = {"git_sha": "abcd12345", "branch": "week2"}
tags = Tags(**tags_dict)

config = ProjectConfig.from_yaml(config_path="../project_config.yml")


In [0]:
# Initialize model
fe_model = FeatureLookUpModel(config=config, tags=tags, spark=spark)

In [0]:
# Create feature table
fe_model.create_feature_table()

2025-09-30 09:58:56.670 | INFO     | hotel_reservation.models.feature_lookup_model:create_feature_table:66 - ✅ Feature table created and populated.


In [0]:
# Define booking value feature function
fe_model.define_feature_function()

2025-09-30 09:59:26.399 | INFO     | hotel_reservation.models.feature_lookup_model:define_feature_function:88 - ✅ Feature function defined.


In [0]:
# Load data
fe_model.load_data()

2025-09-30 10:01:11.817 | INFO     | hotel_reservation.models.feature_lookup_model:load_data:106 - ✅ Data successfully loaded.


In [0]:
# Perform feature engineering
fe_model.feature_engineering()

2025-09-30 10:03:35.095 | INFO     | hotel_reservation.models.feature_lookup_model:feature_engineering:145 - ✅ Feature engineering completed.
2025-09-30 10:03:35.100 | INFO     | hotel_reservation.models.feature_lookup_model:feature_engineering:150 - ✅ Target successfully encoded.


In [0]:
# Train the model
fe_model.train()

2025-09-30 10:04:06.554 | INFO     | hotel_reservation.models.feature_lookup_model:train:157 - 🚀 Starting training...


[LightGBM] [Info] Number of positive: 19551, number of negative: 9469
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.004068 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 915
[LightGBM] [Info] Number of data points in the train set: 29020, number of used features: 29
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.673708 -> initscore=0.725003
[LightGBM] [Info] Start training from score 0.725003
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -

/local_disk0/.ephemeral_nfs/envs/pythonEnv-b39fbe36-4017-4d13-bd75-500c6265efca/lib/python3.12/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
2025-09-30 10:04:10.392 | INFO     | hotel_reservation.models.feature_lookup_model:train:177 - 📊 Accuracy: 0.8780151619572708
2025-09-30 10:04:10.393 | INFO     | hotel_reservation.models.feature_lookup_model:train:178 - 📊 Precision: 0.8893265064986215
2025-09-30 10:04:10.394 | INFO     | hotel_reservation.models.feature_lookup_model:train:179 - 📊 Recall: 0.9332506716263691
/local_disk0/.ephemeral_nfs/envs/pythonEnv-b39fbe36-4017-4d13-bd75-500c6265efca/lib/python3.12/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause 

In [0]:
# Train the model
fe_model.register_model()

Registered model 'mlops_dev.nikhilko.hotel_reservations_model_fe' already exists. Creating a new version of this model...
2025/09/30 10:05:26 WARNING mlflow.tracking._model_registry.fluent: Run with id 9285139a35b24a49b378fd42b5e50081 has no artifacts at artifact path 'lightgbm-pipeline-model-fe', registering model based on models:/m-67e91d158d6b413a9e3c156770e4bda2 instead


Uploading artifacts:   0%|          | 0/15 [00:00<?, ?it/s]

🔗 Created version '2' of model 'mlops_dev.nikhilko.hotel_reservations_model_fe': https://dbc-f122dc18-1b68.cloud.databricks.com/explore/data/models/mlops_dev/nikhilko/hotel_reservations_model_fe/version/2?o=2661948581729539


'2'

In [0]:
# Lets run prediction on the last production model
# Load test set from Delta table
spark = SparkSession.builder.getOrCreate()

test_set = spark.table(f"{config.catalog_name}.{config.schema_name}.test_set").limit(10)

# Drop feature lookup columns and target
X_test = test_set.drop("no_of_previous_cancellations", "no_of_previous_bookings_not_canceled", config.target)


In [0]:
X_test.printSchema()

root
 |-- type_of_meal_plan: string (nullable = true)
 |-- room_type_reserved: string (nullable = true)
 |-- market_segment_type: string (nullable = true)
 |-- no_of_adults: long (nullable = true)
 |-- no_of_children: long (nullable = true)
 |-- no_of_weekend_nights: long (nullable = true)
 |-- no_of_week_nights: long (nullable = true)
 |-- required_car_parking_space: long (nullable = true)
 |-- lead_time: long (nullable = true)
 |-- arrival_year: long (nullable = true)
 |-- arrival_month: long (nullable = true)
 |-- arrival_date: long (nullable = true)
 |-- repeated_guest: long (nullable = true)
 |-- avg_price_per_room: double (nullable = true)
 |-- no_of_special_requests: long (nullable = true)
 |-- Booking_ID: string (nullable = true)
 |-- update_timestamp_utc: timestamp (nullable = true)



In [0]:
from pyspark.sql.functions import col

# X_test = X_test.withColumn("no_of_week_nights", col("no_of_week_nights").cast("int")).withColumn(
#     "no_of_weekend_nights", col("no_of_weekend_nights").cast("int")
# )

In [0]:
fe_model = FeatureLookUpModel(config=config, tags=tags, spark=spark)

# Make predictions
predictions = fe_model.load_latest_model_and_predict(X_test)

# Display predictions
logger.info(predictions)

2025/09/30 10:06:35 WARNING mlflow.pyfunc: Calling `spark_udf()` with `env_manager="local"` does not recreate the same environment that was used during training, which may lead to errors or inaccurate predictions. We recommend specifying `env_manager="conda"`, which automatically recreates the environment that was used to train the model and performs inference in the recreated environment.


2025/09/30 10:06:35 INFO mlflow.models.flavor_backend_registry: Selected backend for flavor 'python_function'
2025-09-30 10:06:39.470 | INFO     | __main__:<module>:7 - DataFrame[type_of_meal_plan: string, room_type_reserved: string, market_segment_type: string, no_of_adults: bigint, no_of_children: bigint, no_of_weekend_nights: bigint, no_of_week_nights: bigint, required_car_parking_space: bigint, lead_time: bigint, arrival_year: bigint, arrival_month: bigint, arrival_date: bigint, repeated_guest: bigint, avg_price_per_room: double, no_of_special_requests: bigint, Booking_ID: string, update_timestamp_utc: timestamp, no_of_previous_cancellations: bigint, no_of_previous_bookings_not_canceled: bigint, booking_value: double, prediction: double]


In [0]:
display(predictions)

type_of_meal_plan,room_type_reserved,market_segment_type,no_of_adults,no_of_children,no_of_weekend_nights,no_of_week_nights,required_car_parking_space,lead_time,arrival_year,arrival_month,arrival_date,repeated_guest,avg_price_per_room,no_of_special_requests,Booking_ID,update_timestamp_utc,no_of_previous_cancellations,no_of_previous_bookings_not_canceled,booking_value,prediction
Meal Plan 1,Room_Type 1,Offline,2,0,1,2,0,305,2018,11,4,0,89.0,0,INN15336,2025-09-07T15:47:19.339Z,0,0,267.0,0.0
Meal Plan 1,Room_Type 4,Online,3,0,1,4,0,211,2018,8,22,0,121.55,2,INN33915,2025-09-07T15:47:19.339Z,0,0,607.75,0.0
Meal Plan 1,Room_Type 1,Online,2,0,2,3,0,63,2017,10,10,0,89.25,0,INN11389,2025-09-07T15:47:19.339Z,0,0,446.25,1.0
Not Selected,Room_Type 1,Online,2,0,0,2,0,163,2018,12,2,0,79.2,0,INN10762,2025-09-07T15:47:19.339Z,0,0,158.4,0.0
Meal Plan 1,Room_Type 4,Online,2,0,0,4,0,40,2018,12,7,0,96.9,1,INN16234,2025-09-07T15:47:19.339Z,0,0,387.6,1.0
Meal Plan 1,Room_Type 1,Online,2,0,0,2,0,48,2018,2,11,0,89.3,0,INN28112,2025-09-07T15:47:19.339Z,0,0,178.6,0.0
Meal Plan 1,Room_Type 1,Corporate,1,0,1,0,0,27,2017,11,29,0,65.0,0,INN17698,2025-09-07T15:47:19.339Z,0,0,65.0,1.0
Meal Plan 1,Room_Type 1,Online,2,0,2,2,0,28,2018,5,1,0,133.95,1,INN23760,2025-09-07T15:47:19.339Z,0,0,535.8,1.0
Meal Plan 1,Room_Type 1,Offline,2,0,0,2,0,42,2018,2,26,0,74.0,0,INN22546,2025-09-07T15:47:19.339Z,0,0,148.0,1.0
Meal Plan 1,Room_Type 4,Online,2,0,2,1,0,1,2017,10,4,0,136.0,0,INN19133,2025-09-07T15:47:19.339Z,0,0,408.0,1.0
